# Lab 03 — Model Representation and Cost
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 3 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab03-model-and-cost/Lab03_Model_and_Cost.ipynb)

In Lab 02 you saw that stronger concrete tends to have a lower water–cement ratio. Today you build the simplest machine-learning model — a **straight line** — and, more importantly, a way to *measure* how good a line is: the **cost function**. Then you search for the line with the lowest cost. This model → cost → search recipe is how every model in this course learns, up to large language models. Questions marked 🏠 may be finished after lab; they are still part of the lab.

### What you will be able to do
1. Write the linear model f(x) = w·x + b as a Python function and explain w and b with units.
2. Compute residuals and the cost J(w, b) = (1/2m) Σ (f(x⁽ⁱ⁾) − y⁽ⁱ⁾)² by hand and in code.
3. Plot the cost as a function of w (a bowl) and of w and b together (a contour map).
4. Find a good (w, b) by grid search and check it against `numpy.polyfit`.
5. Explain the limits of a straight line for concrete strength (Abrams' law, extrapolation).

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `concrete_strength.csv` — 1,030 laboratory concrete tests (Yeh 1998, UCI, CC BY 4.0).

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — The data and the notation (10 min)
We predict compressive **strength** (MPa) from **one feature**: the water–cement ratio (w/c), the column you built
in Lab 02. `.to_numpy()` turns a pandas column into a numpy array, so we can do fast math on it as in Lab 01.

In [ ]:
df = pd.read_csv(get_data("concrete_strength.csv"))
df["wc_ratio"] = df["water_kg_m3"] / df["cement_kg_m3"]

x = df["wc_ratio"].to_numpy()       # feature: water-cement ratio (–)
y = df["strength_mpa"].to_numpy()   # target: compressive strength (MPa)
m = len(x)                          # number of training examples
print("m =", m)

**Notation** (the same as in the lecture):

| Symbol | Meaning | Here |
|---|---|---|
| x⁽ⁱ⁾ | feature of example *i* (the ⁽ⁱ⁾ is a label, **not** a power) | w/c of test *i* |
| y⁽ⁱ⁾ | target of example *i* | measured strength of test *i* (MPa) |
| m | number of examples | 1,030 |
| ŷ = f(x) | the model's prediction ("y-hat") | predicted strength (MPa) |

Python counts from 0, so example 1 (x⁽¹⁾) is `x[0]`.

In [ ]:
for i in range(3):
    print(f"Example {i + 1}: w/c = {x[i]:.3f},  strength = {y[i]:.1f} MPa")

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x, y, alpha=0.35, s=12)
plt.xlabel("Water–cement ratio, x (–)")
plt.ylabel("Compressive strength, y (MPa)")
plt.title("1,030 concrete tests: strength vs w/c")
plt.show()

## Part 2 — The model: a straight line (15 min)
$$\hat{y} = f_{w,b}(x) = w \cdot x + b$$

- **w** (the weight, or slope) — how much the predicted strength changes when w/c goes up by 1. Since w/c has no
  units, w is in **MPa**. A w/c change of 0.1 changes the prediction by w/10.
- **b** (the bias, or intercept) — the prediction when x = 0, in **MPa**. Concrete with zero water does not exist,
  so b has no physical meaning by itself; it just positions the line.

**Learning** means choosing the numbers w and b from data.

> 💡 **Hint:** Translate the equation directly: `w * x + b`.

In [ ]:
# TODO: complete the model. It must work for one number AND for a whole numpy array x.
def predict(x, w, b):
    """Predicted strength (MPa) for water-cement ratio x, with slope w (MPa) and intercept b (MPa)."""
    return None   # replace None

In [ ]:
# ✅ Self-check: run this cell to test your work above
r = predict(0.5, -30, 60)
assert r is not None, "predict returned None. Finish the TODO above first: replace None with w * x + b."
assert r == 45, f"predict(0.5, -30, 60) should be -30 × 0.5 + 60 = 45 MPa, but it gave {r}. Check the formula w * x + b."
assert np.shape(predict(np.array([0.4, 0.6]), -30, 60)) == (2,), "predict should give one prediction per x value in an array."
print("Correct!")

This helper draws the data and the line for any (w, b). Run it — it does not need changes.

In [ ]:
def plot_line(w, b):
    """Scatter the data and draw the line f(x) = w*x + b."""
    x_line = np.linspace(0.2, 2.0, 100)
    plt.figure(figsize=(7, 4))
    plt.scatter(x, y, alpha=0.3, s=12, label="tests")
    plt.plot(x_line, predict(x_line, w, b), color="red", label=f"f(x) = {w:.1f}·x + {b:.1f}")
    plt.xlabel("Water–cement ratio, x (–)")
    plt.ylabel("Compressive strength (MPa)")
    plt.title("Concrete strength vs w/c, with a candidate line")
    plt.ylim(-10, 90)
    plt.legend()
    plt.show()

# A first try: a flat line at the average strength ("predict the average for every mix")
plot_line(0, 35.8)

> 💡 **Hint:** The cloud slopes downward, so w must be negative. Start with w = −20 and b = 50, then adjust one number at a time.

In [ ]:
# TODO: try different values of w and b until the line follows the data.
#       Re-run this cell after each change. Try at least three pairs; keep your best one.
my_w = 0       # MPa
my_b = 35.8    # MPa
plot_line(my_w, my_b)

**Optional — sliders.** In Colab, the cell below shows sliders for w and b; drag them to move the line. If you see an error instead, just skip it.

In [ ]:
try:
    from ipywidgets import interact
    interact(plot_line, w=(-80.0, 20.0, 1.0), b=(0.0, 100.0, 1.0))
except ImportError:
    print("Sliders are not available here — use the cell above instead.")

**✍️ Question:** Write your best line in words with units: "Predicted strength drops by about ___ MPa for each 0.1 increase in w/c." Does that sound reasonable to you as an engineer?

*Your answer:* 

## Part 3 — How wrong is a line? Residuals and cost (20 min)
For each example, the **error** (or **residual**) is the prediction minus the measured value:
f(x⁽ⁱ⁾) − y⁽ⁱ⁾. Positive = the line predicts too high; negative = too low. The red segments below are the errors
of 25 randomly chosen tests for your line.

In [ ]:
rng = np.random.default_rng(42)                 # random numbers with a fixed seed (same every run)
idx = rng.choice(m, size=25, replace=False)     # 25 random row numbers

plt.figure(figsize=(7, 4))
plt.scatter(x[idx], y[idx], color="navy", s=20, label="measured")
x_line = np.linspace(0.2, 2.0, 100)
plt.plot(x_line, predict(x_line, my_w, my_b), color="red", label="your line")
for i in idx:
    plt.plot([x[i], x[i]], [y[i], predict(x[i], my_w, my_b)], color="red", alpha=0.5, linewidth=1)
plt.xlabel("Water–cement ratio, x (–)")
plt.ylabel("Compressive strength (MPa)")
plt.title("Errors (residuals) for 25 random tests")
plt.legend()
plt.show()

We need **one number** that says how wrong the line is over *all* examples. That number is the
**cost function**:

$$J(w,b) = \frac{1}{2m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)^2$$

Read it from the inside out: (1) predict, (2) subtract the true value → error, (3) **square** it, (4) **add** over
all m examples (Σ means "sum"), (5) divide by 2m.

- **Why square?** So positive and negative errors cannot cancel, and big errors count much more than small ones.
- **Why divide by m?** To get an *average*, so the cost does not grow just because we have more data.
- **Why the extra ½?** It makes next week's slope formula simpler. It does not change *which* line is best.
- **Units:** MPa² (squared MPa). Lower is better; J = 0 would mean every prediction is exact.

**Worked example by hand** — the same 3 mixes and Line 1 as in the Week 3 lecture (w = −50 MPa, b = 70 MPa):

| i | x⁽ⁱ⁾ | y⁽ⁱ⁾ (MPa) | f(x⁽ⁱ⁾) = −50·x + 70 | error | error² |
|---|---|---|---|---|---|
| 1 | 0.4 | 50 | 50 | 0 | 0 |
| 2 | 0.6 | 38 | 40 | 2 | 4 |
| 3 | 0.8 | 30 | 30 | 0 | 0 |

Sum of squared errors = 4, m = 3, so **J = 4 / (2·3) ≈ 0.67 MPa²** — the lecture's answer.

> 💡 **Hint:** Because x and y are numpy arrays, `predict(x, w, b) - y` gives all errors at once. `np.sum(errors**2)` adds up the squared errors. **Parentheses!** Write `np.sum(errors**2) / (2 * m)`. Without them, `/ 2 * m` divides by 2 and then *multiplies* by m.

In [ ]:
# TODO: complete the cost function (three lines).
def compute_cost(x, y, w, b):
    """Cost J(w, b) = (1/2m) * sum of squared errors, in MPa²."""
    m = len(x)
    f = None        # predictions for all examples (use predict)
    errors = None   # prediction minus measured value
    J = None        # sum of squared errors divided by (2 * m) — keep the parentheses!
    return J

In [ ]:
# ✅ Self-check: run this cell to test your work above
x3 = np.array([0.4, 0.6, 0.8]); y3 = np.array([50.0, 38.0, 30.0])   # the lecture's 3 mixes
J3 = compute_cost(x3, y3, -50, 70)
assert J3 is not None, "compute_cost returned None. Finish the three TODO lines above (each one still says None)."
assert abs(J3 - 6.0) > 1e-9, "You got 6.0: '/ 2 * m' divides by 2 and then MULTIPLIES by m. Write np.sum(errors**2) / (2 * m)."
assert abs(J3 - 4 / 3) > 1e-9, "You got 1.333: that divides by m only. The cost divides by 2m."
assert abs(J3 - 4 / 6) < 1e-9, f"Expected 0.667 MPa² for the 3 mixes, but got {J3:.3f}. Check errors = f - y and the square (**2)."
print(f"Correct! J = {J3:.2f} MPa² — the same as the hand calculation in the lecture.")

> 💡 **Hint:** Call `compute_cost(x, y, w, b)` three times with different w and b.

In [ ]:
# TODO: compute the cost (on all 1,030 tests) of three lines and print them.
J_flat = None   # w = 0, b = the average strength np.mean(y)
J_mine = None   # your my_w, my_b from Part 2
J_bad = None    # w = 20, b = 20 (a line that slopes the wrong way)
print(f"Flat line: J = {J_flat}   Your line: J = {J_mine}   Wrong-way line: J = {J_bad}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert None not in (J_flat, J_mine, J_bad), "A cost is still None. Finish the TODO above first."
assert abs(J_flat - 139.4) < 0.1, f"J_flat should be about 139.4 MPa², got {J_flat:.1f}. Use w = 0 and b = np.mean(y)."
assert abs(J_bad - 211.9) < 0.1, f"J_bad should be about 211.9 MPa², got {J_bad:.1f}. Use w = 20 and b = 20."
assert J_mine < J_flat, f"Your line (J = {J_mine:.1f}) should beat the flat line (139.4). Go back to Part 2 and adjust my_w and my_b."
print("Correct!")

MPa² is hard to feel. Undo the square: $\sqrt{2J}$ is the **root-mean-square error (RMSE)** in MPa — a
typical size of the prediction error.

In [ ]:
for name, J in [("flat line", J_flat), ("your line", J_mine), ("wrong-way line", J_bad)]:
    print(f"{name:15s} J = {J:6.1f} MPa²   typical error sqrt(2J) = {np.sqrt(2 * J):.1f} MPa")

**✍️ Question:** Your best line has a typical error of about 14–15 MPa. Specified strengths for buildings are often 20–40 MPa. Would you use this line to decide whether a mix is strong enough for a column? Why or why not?

*Your answer:* 

## Part 4 — The cost landscape (15 min)
### 4.1 Cost as a function of w
Fix b = 55 MPa and try many values of w. For each w, compute the cost. `np.linspace(-80, 30, 111)` makes 111
evenly spaced numbers from −80 to 30 (steps of 1).

> 💡 **Hint:** Inside the loop: `costs.append(compute_cost(x, y, w, b_fixed))`. Delete `pass`.

In [ ]:
b_fixed = 55.0                         # MPa
w_values = np.linspace(-80, 30, 111)   # MPa
costs = []
for w in w_values:
    # TODO: compute the cost for this w (with b = b_fixed) and append it to the list costs
    pass

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert len(costs) > 0, "costs is still empty. Inside the loop, add costs.append(compute_cost(x, y, w, b_fixed)) and delete pass."
assert len(costs) == 111, f"costs should have one value per w (111), but it has {len(costs)}. Did you run the cell more than once? Re-run it from the line costs = []."
print("Correct!")

`np.argmin(costs)` gives the **position** of the smallest value in the list; we use it to find the bottom.

In [ ]:
best_i = np.argmin(costs)
plt.figure(figsize=(7, 4))
plt.plot(w_values, costs)
plt.scatter([w_values[best_i]], [costs[best_i]], color="red", zorder=3,
            label=f"lowest: w = {w_values[best_i]:.0f} MPa, J = {costs[best_i]:.1f} MPa²")
plt.xlabel("w (MPa)")
plt.ylabel("Cost J (MPa²)")
plt.title(f"Cost vs w, with b fixed at {b_fixed:.0f} MPa")
plt.legend()
plt.show()

**✍️ Question:** Describe the shape of the curve. Roughly where is the bottom? What happens to the cost as w moves away from the bottom in either direction?

*Your answer:* 

### 4.2 Cost as a function of w and b together
Now vary **both** w and b. We fill a table `J_grid` with one cost for every (w, b) pair. Two new tools:
- `np.zeros((rows, cols))` makes a table of zeros to fill in;
- `enumerate(...)` gives both the position (`i`) and the value (`b`) in a loop.

This computes about 8,000 costs — it takes a few seconds.

In [ ]:
w_grid = np.linspace(-80, 30, 111)   # MPa (steps of 1)
b_grid = np.linspace(20, 90, 71)     # MPa (steps of 1)
J_grid = np.zeros((len(b_grid), len(w_grid)))   # rows = b values, columns = w values

for i, b in enumerate(b_grid):
    for j, w in enumerate(w_grid):
        J_grid[i, j] = compute_cost(x, y, w, b)

print("Smallest cost on the grid:", round(J_grid.min(), 2), "MPa²")

Two pictures of the same table: a **3-D bowl**, and a **contour map** — like a topographic map, where each
line connects (w, b) pairs with the *same* cost. The bottom of the bowl is the center of the smallest ring.

In [ ]:
W, B = np.meshgrid(w_grid, b_grid)              # 2-D versions of the grids, for plotting
levels = [105, 108, 115, 130, 160, 220, 350, 600, 1000, 1600, 2500]   # MPa²

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(W, B, J_grid, cmap="viridis", alpha=0.9)
ax1.set_xlabel("w (MPa)"); ax1.set_ylabel("b (MPa)"); ax1.set_zlabel("J (MPa²)")
ax1.set_title("Cost surface J(w, b)")

ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contour(W, B, J_grid, levels=levels, cmap="viridis")
ax2.clabel(cs, fontsize=8, fmt="%.0f")
ax2.set_xlabel("w (MPa)"); ax2.set_ylabel("b (MPa)")
ax2.set_title("Contour map of J (MPa²)")
plt.tight_layout()
plt.show()

### 4.3 Grid search
Find the smallest value in `J_grid` with loops — the idea is the same as finding the smallest number in a list by
hand: keep the best so far, and replace it whenever you find something smaller.

> 💡 **Hint:** Use an `if` with `<`, then three assignments under it: `best_J = J_grid[i, j]`, `best_w = w`, `best_b = b`. Lines that belong to the `if` start 4 spaces further right than the `if` (an `IndentationError` means the spaces do not line up). Delete `pass`.

In [ ]:
best_J = float("inf")     # start with "infinitely bad"
best_w, best_b = None, None
for i, b in enumerate(b_grid):
    for j, w in enumerate(w_grid):
        # TODO: if J_grid[i, j] is smaller than best_J, store it in best_J and remember this w and b
        pass
print(f"Grid search: w = {best_w} MPa, b = {best_b} MPa, J = {best_J} MPa²")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert best_w is not None, "best_w is still None. Inside the loop, the if-block must store w and b when J_grid[i, j] < best_J."
assert abs(best_w - (-27)) <= 1 and abs(best_b - 56) <= 1,     f"Expected about w = -27 MPa and b = 56 MPa, but got w = {best_w}, b = {best_b}. Is the comparison '<' (smaller), and do you store both w and b?"
print("Correct!")

In [ ]:
plt.figure(figsize=(7, 5))
cs = plt.contour(W, B, J_grid, levels=levels, cmap="viridis")
plt.clabel(cs, fontsize=8, fmt="%.0f")
plt.scatter([best_w], [best_b], color="red", marker="*", s=200, zorder=3, label="grid-search best")
plt.xlabel("w (MPa)"); plt.ylabel("b (MPa)")
plt.title("Contour map of J (MPa²) with the grid-search minimum")
plt.legend()
plt.show()

**Counting grid points.** Your grid has 111 values of w × 71 values of b = 7,881 points. With steps
10 times smaller in **both** w and b, there are 10× more values of w *and* 10× more values of b, so
10 × 10 = **100× more points** (about 790,000). Each extra number to choose multiplies the count again.

**✍️ Question:** 🏠 **Finish at home** (still part of the lab, due Sunday). (a) Suppose a model has **3** numbers to choose (2 weights + b), and you try 100 values of each. How many grid points is that? (b) A model with 8 features has **9** numbers to choose (8 weights + b). With 100 values for each, how many grid points? (c) What does this tell you about grid search?

*Your answer:* 

## Part 5 — Check with `numpy.polyfit`, and the limits of a straight line (10 min)
`np.polyfit(x, y, 1)` finds the line with the **lowest possible** squared error directly, using linear algebra
(no grid). The `1` means degree 1 = a straight line. It returns `[slope, intercept]`, which you can unpack into two
variables at once.

> 💡 **Hint:** `w_fit, b_fit = np.polyfit(x, y, 1)` then `J_fit = compute_cost(x, y, w_fit, b_fit)`.

In [ ]:
# TODO: use np.polyfit to get the best w and b, compute their cost, and print them next to your grid result.
w_fit, b_fit = None, None
J_fit = None
print(f"polyfit:     w = {w_fit}, b = {b_fit}, J = {J_fit}")
print(f"grid search: w = {best_w}, b = {best_b}, J = {best_J}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert w_fit is not None and J_fit is not None, "w_fit or J_fit is still None. Finish the TODO above first."
assert abs(w_fit - (-26.64)) < 0.05 and abs(b_fit - 55.75) < 0.05,     f"Expected w ≈ -26.64 and b ≈ 55.75, got {w_fit:.2f} and {b_fit:.2f}. Use np.polyfit(x, y, 1): x first, then y, then 1."
print("Correct! Grid search was within one grid step of the exact answer.")

**Engineering-friendly measures of fit.** The cost is what the computer minimizes; people usually report:
- **MAE** (mean absolute error) $= \frac{1}{m}\sum |f(x^{(i)}) - y^{(i)}|$ — the average size of the error, in MPa.
- **R²** (coefficient of determination) — the fraction of the variation in strength that the model explains:
  1 = perfect, 0 = no better than predicting the average for everyone.

In [ ]:
f_fit = predict(x, w_fit, b_fit)
mae = np.mean(np.abs(f_fit - y))
r2 = 1 - np.sum((f_fit - y)**2) / np.sum((y - np.mean(y))**2)
print(f"On all 1,030 tests: MAE = {mae:.1f} MPa,  RMSE = {np.sqrt(2 * J_fit):.1f} MPa,  R² = {r2:.2f}")

**✍️ Question:** 🏠 **Finish at home** (still part of the lab, due Sunday). R² ≈ 0.25. In plain words, what does that mean? From what you saw in Lab 02, what else explains the rest of the variation in strength?

*Your answer:* 

### The true relationship is curved
In 1918, Duff Abrams reported that concrete strength falls roughly **exponentially** as w/c rises:
strength ≈ A / B^(w/c), where A and B depend on the materials and the age ("Abrams' law"). Below we fit that curve
for you (with `scipy`), and also plot the **average strength in w/c bins** (large dots) so the shape is easier to
see through the scatter. We extend the x-axis beyond the data on purpose.

In [ ]:
from scipy.optimize import curve_fit

def abrams(x, A, B):
    return A / B**x

(A, B_abrams), _ = curve_fit(abrams, x, y, p0=(60, 2.4))
f_abrams = abrams(x, A, B_abrams)
r2_abrams = 1 - np.sum((f_abrams - y)**2) / np.sum((y - np.mean(y))**2)
print(f"Abrams curve: A = {A:.1f} MPa, B = {B_abrams:.2f},  R² = {r2_abrams:.2f}  (line: R² = {r2:.2f})")

# average strength in w/c bins
bins = [0.25, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.4, 1.9]
binned = df.groupby(pd.cut(df["wc_ratio"], bins), observed=True).agg(wc=("wc_ratio", "mean"),
                                                                      strength=("strength_mpa", "mean"))

x_line = np.linspace(0.2, 2.4, 200)
plt.figure(figsize=(8, 4.5))
plt.scatter(x, y, alpha=0.2, s=10, label="tests")
plt.scatter(binned["wc"], binned["strength"], color="black", s=60, zorder=3, label="average in each w/c bin")
plt.plot(x_line, predict(x_line, w_fit, b_fit), color="red", label="best straight line")
plt.plot(x_line, abrams(x_line, A, B_abrams), color="green", label="Abrams-law curve")
plt.axvspan(x.max(), 2.4, color="gray", alpha=0.15, label="beyond the data")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Water–cement ratio (–)")
plt.ylabel("Compressive strength (MPa)")
plt.title("A straight line vs Abrams' law")
plt.legend(fontsize=9)
plt.show()

print(f"At w/c = 2.2 the line predicts {predict(2.2, w_fit, b_fit):.1f} MPa; the curve predicts {abrams(2.2, A, B_abrams):.1f} MPa.")

**✍️ Question:** 🏠 **Finish at home** (still part of the lab, due Sunday). (a) Where do the bin averages bend away from the straight line? (b) What does the line predict at w/c = 2.2, and why is that impossible? (c) The curve only improves R² a little (≈0.28 vs ≈0.25). Why doesn't the better shape help more?

*Your answer:* 

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** In f(x) = w·x + b for strength vs water–cement ratio, what are the units of w and of b?

*Your answer:* 

**2.** What does the cost J(w, b) measure, and why do we square the errors?

*Your answer:* 

**3.** Grid search worked for 2 numbers (w and b). Why won't it work for a model with 9 numbers to choose?

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.